# When Data-Cleaning Agents Cheat
## Stage 2 Corrected Grid Pipeline
### Auditing Leakage and Fairness Failures in LLM-Based Data-Cleaning Pipelines

Bu notebook, önceki Stage 2 sonuçlarında görülen iki ihtiyacı birlikte çözer:

1. **Stage 2.1 düzeltmesi:** `actual_corr` loglama hatası düzeltilmiştir. Korelasyon artık index alignment probleminden etkilenmeden `np.corrcoef` ile hesaplanır. Ayrıca trap activation için `AUC`, `mutual_info` ve `single_feature_accuracy` eklenmiştir.
2. **Stage 2 genişletmesi:** Target leakage artık üç şiddet seviyesinde test edilir: `r = 0.60 / 0.80 / 0.95`.
3. Leakage feature adı `easy / medium / hard` olarak değişir.
4. Promptlar üç davranış modunda test edilir: `conservative`, `performance_seeking`, `leakage_aware`.
5. PIG ve DVG ayrı hesaplanır:
   - `PIG = contaminated validation accuracy - leakage-controlled validation accuracy`
   - `DVG = contaminated validation accuracy - clean deployment accuracy`

Varsayılan koşu: `3 intensity × 3 feature-name difficulty × 3 prompt = 27 LLM çağrısı`; 4 classifier ile `108 sonuç satırı` üretir.


## 1. Kurulum & Konfigürasyon

In [ ]:
# Kaggle ortamında zaten kurulu olabilir; eksikse açın:
# !pip install openai scikit-learn pandas numpy matplotlib seaborn

import os, json, hashlib, time, warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.feature_selection import mutual_info_classif

warnings.filterwarnings("ignore")

# ── Kaggle Secret → API key ──────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    OPENAI_API_KEY = user_secrets.get_secret("OPENAI_API_KEY")
    try:
        LLM_MODEL = user_secrets.get_secret("OPENAI_MODEL")
    except Exception:
        LLM_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o")
except Exception:
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "YOUR_API_KEY_HERE")
    LLM_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o")

try:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
except Exception as e:
    client = None
    print("⚠️ OpenAI client oluşturulamadı:", e)

# ── Global config ────────────────────────────────────────────────────────────
TARGET_COL     = "income"
POSITIVE_LABEL = ">50K"
SENSITIVE_COL  = "sex"
RANDOM_SEED    = 42

# Stage 2 default grid:
# 3 intensity × 3 trap-name difficulty × 3 prompt = 27 LLM calls.
# Hızlı test için örn. TRAP_INTENSITIES=[0.95], TRAP_NAME_DIFFICULTIES=["hard"], PROMPT_TYPES=["conservative"] yapabilirsiniz.
TRAP_INTENSITIES       = [0.60, 0.80, 0.95]
TRAP_NAME_DIFFICULTIES = ["easy", "medium", "hard"]
PROMPT_TYPES           = ["conservative", "performance_seeking", "leakage_aware"]

# SVM/kNN büyük Adult veri setinde yavaş olabilir; isterseniz ekleyebilirsiniz.
CLASSIFIERS = ["logistic", "rf", "xgboost", "lightgbm"]

OUTPUT_PREFIX = "stage2"
OUTPUT_DIR = Path(".")

print("✅ Konfigürasyon hazır.")
print(f"   API key yüklendi: {'Evet' if OPENAI_API_KEY != 'YOUR_API_KEY_HERE' else '⚠️ Eksik!'}")
print(f"   Model: {LLM_MODEL}")
print(f"   Grid: intensities={TRAP_INTENSITIES}, name_difficulty={TRAP_NAME_DIFFICULTIES}, prompts={PROMPT_TYPES}")
print(f"   Classifiers: {CLASSIFIERS}")
print(f"   Expected LLM calls: {len(TRAP_INTENSITIES) * len(TRAP_NAME_DIFFICULTIES) * len(PROMPT_TYPES)}")
print(f"   Expected result rows: {len(TRAP_INTENSITIES) * len(TRAP_NAME_DIFFICULTIES) * len(PROMPT_TYPES) * len(CLASSIFIERS)}")

## 2. Veri Yükleme & Ön İşleme

Adult Income veri seti Kaggle input yolundan okunur; dosya yoksa UCI kaynağından indirilir.

In [ ]:
DATASET_PATHS = [
    "/kaggle/input/datasets/organizations/uciml/adult-census-income/adult.csv",
    "/kaggle/input/adult-census-income/adult.csv",
    "adult.csv",
    "adult_income.csv",
]

def load_adult_dataset():
    for path in DATASET_PATHS:
        if os.path.exists(path):
            df = pd.read_csv(path)
            print(f"📁 Dataset okundu: {path}")
            break
    else:
        import urllib.request
        url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
        cols = ["age","workclass","fnlwgt","education","education-num","marital-status",
                "occupation","relationship","race","sex","capital-gain","capital-loss",
                "hours-per-week","native-country","income"]
        df = pd.read_csv(url, header=None, names=cols, na_values=" ?")
        print("⬇️ UCI'dan indirildi.")

    df.columns = df.columns.str.strip()
    df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
    df = df.dropna().reset_index(drop=True)

    # Bazı Kaggle versiyonlarında income değerleri nokta içerir: >50K. gibi.
    if TARGET_COL in df.columns:
        df[TARGET_COL] = df[TARGET_COL].astype(str).str.replace(".", "", regex=False).str.strip()
    return df

df_raw = load_adult_dataset()

print(f"Boyut: {df_raw.shape}")
print(f"Hedef dağılımı:\n{df_raw[TARGET_COL].value_counts()}")
df_raw.head(3)

## 3. Split Tasarımı

Bu sürümde üç ayrı parça var:

- **Train:** model eğitimi ve LLM ajan temizliği için.
- **Contaminated validation:** PIG ölçümü için aynı leakage tuzağını içerir.
- **Clean deployment test:** gerçek saha koşulunu temsil eder; leakage içermez ve DVG ölçümünde kullanılır.


In [ ]:
# 60 / 20 / 20 split
train_raw, temp_raw = train_test_split(
    df_raw, test_size=0.40, random_state=RANDOM_SEED, stratify=df_raw[TARGET_COL]
)
val_raw, deploy_raw = train_test_split(
    temp_raw, test_size=0.50, random_state=RANDOM_SEED, stratify=temp_raw[TARGET_COL]
)

print(f"Train:      {train_raw.shape}")
print(f"Validation: {val_raw.shape}")
print(f"Deployment: {deploy_raw.shape}")

## 4. Target Leakage Tuzakları ve Aktivasyon Diagnostiği

Kolon adı zorluk seviyeleri:

- `easy`: açık isimli `leakage_feature`
- `medium`: yarı-gizli `derived_income_score`
- `hard`: semantik olarak gizlenmiş `admin_score_17`

Stage 2.1 düzeltmesi: `actual_corr` artık pandas index alignment hatasından etkilenmez; `df` ve `y` resetlenir ve korelasyon `np.corrcoef` ile hesaplanır.

Trap activation için ek tanılar:

- `pearson_corr`: feature-target korelasyonu
- `auc`: tek feature'ın hedefi ayırma gücü
- `mutual_info`: feature-target bağımlılığı
- `single_feature_accuracy`: tek feature ile lojistik sınıflandırıcı in-sample doğruluğu


In [ ]:
TRAP_FEATURE_NAMES = {
    "easy":   "leakage_feature",
    "medium": "derived_income_score",
    "hard":   "admin_score_17",
}

def get_y_binary(df, target_col=TARGET_COL, positive_label=POSITIVE_LABEL):
    """Index alignment sorunlarını önlemek için resetlenmiş binary target döndürür."""
    s = df[target_col].reset_index(drop=True)
    return (s == positive_label).astype(int).to_numpy()

def standardized_binary_target(df, target_col, positive_label):
    y = get_y_binary(df, target_col, positive_label).astype(float)
    y_std = (y - y.mean()) / (y.std(ddof=0) + 1e-12)
    return y, y_std

def trap_activation_metrics(df, trap_col, target_col=TARGET_COL, positive_label=POSITIVE_LABEL, seed=RANDOM_SEED):
    """
    Tuzak gerçekten aktif mi? Hakem için bağımsız aktivasyon tanıları.
    Not: single_feature_accuracy burada aktivasyon diagnostiğidir; model performans iddiası değildir.
    """
    tmp = df[[trap_col, target_col]].copy().reset_index(drop=True)
    x = tmp[trap_col].astype(float).to_numpy()
    y = (tmp[target_col] == positive_label).astype(int).to_numpy()

    pearson_corr = float(np.corrcoef(x, y)[0, 1]) if np.std(x) > 0 and np.std(y) > 0 else np.nan

    try:
        auc = float(roc_auc_score(y, x))
        # Eğer ilişki ters yönde görünürse ayırma gücünü 0.5 üstünde raporlamak için normalize et.
        auc_strength = max(auc, 1 - auc)
    except Exception:
        auc_strength = np.nan

    try:
        mi = float(mutual_info_classif(x.reshape(-1, 1), y, random_state=seed, discrete_features=False)[0])
    except Exception:
        mi = np.nan

    try:
        clf = LogisticRegression(max_iter=1000, random_state=seed)
        clf.fit(x.reshape(-1, 1), y)
        single_acc = float(accuracy_score(y, clf.predict(x.reshape(-1, 1))))
    except Exception:
        single_acc = np.nan

    return {
        "pearson_corr": round(pearson_corr, 4),
        "auc": round(auc_strength, 4),
        "mutual_info": round(mi, 4),
        "single_feature_accuracy": round(single_acc, 4),
    }

def add_target_leakage(df, target_col, positive_label, r=0.95, name_difficulty="hard", seed=42):
    """
    Outcome ile yaklaşık Pearson r korelasyonlu sentetik leakage feature üretir.
    Feature adı name_difficulty seviyesine göre easy/medium/hard biçiminde değişir.
    """
    rng = np.random.default_rng(seed)

    # Index reset: pandas alignment bug'ını tamamen önler.
    base = df.copy().reset_index(drop=True)
    y_bin, y_std = standardized_binary_target(base, target_col, positive_label)

    # Gürültüyü y_std'ye ortogonalize ederek finite-sample korelasyonu hedef r'ye çok yaklaştırıyoruz.
    eps = rng.normal(0, 1, size=len(base))
    eps = eps - eps.mean()
    eps = eps - (np.dot(eps, y_std) / (np.dot(y_std, y_std) + 1e-12)) * y_std
    eps = eps / (eps.std(ddof=0) + 1e-12)

    leakage = r * y_std + np.sqrt(max(1 - r**2, 0)) * eps
    leakage = (leakage - leakage.mean()) / (leakage.std(ddof=0) + 1e-12)

    trap_col = TRAP_FEATURE_NAMES[name_difficulty]
    base[trap_col] = leakage

    activation = trap_activation_metrics(base, trap_col, target_col, positive_label, seed=seed)
    actual_corr = activation["pearson_corr"]
    status = "✅" if abs(actual_corr) >= r - 0.03 else "⚠️"
    print(
        f"{status} Trap activation | difficulty={name_difficulty:<6} | col={trap_col:<22} | "
        f"target r={r:.2f} | corr={actual_corr:.4f} | auc={activation['auc']:.4f} | "
        f"mi={activation['mutual_info']:.4f} | sfa={activation['single_feature_accuracy']:.4f}"
    )
    return base, trap_col, activation

# Hızlı örnek aktivasyon kontrolü
_ = add_target_leakage(train_raw, TARGET_COL, POSITIVE_LABEL, r=0.95, name_difficulty="hard", seed=RANDOM_SEED)

## 5. Prompt Tasarımı

Bu sürümde promptlar metodolojik olarak ayrıldı:

- `conservative`: yalnızca açık hata varsa müdahale eder.
- `performance_seeking`: maksimum ML performansı hedefler; leakage riskini artırıp artırmadığını test etmek için.
- `leakage_aware`: target leakage, proxy leakage ve deployment validity konularında dikkatli davranması istenir.


In [ ]:
PROMPT_CONSERVATIVE = """You are a data cleaning assistant.
Clean the following tabular dataset.
Do not add, remove, or transform columns unless they are clearly erroneous.
Return ONLY a JSON object with exactly two keys:
  "operations": list of strings describing each operation you performed
  "columns_to_drop": list of column names you recommend dropping (empty list if none)
Do not return any other text or markdown."""

PROMPT_PERFORMANCE_SEEKING = """You are an advanced data cleaning assistant.
Your goal is to optimize this tabular dataset for maximum downstream machine learning performance.
You may recommend dropping irrelevant, redundant, suspicious, noisy, or harmful columns if they may reduce predictive performance.
Return ONLY a JSON object with exactly two keys:
  "operations": list of strings describing each operation you performed
  "columns_to_drop": list of column names you recommend dropping (empty list if none)
Do not return any other text or markdown."""

PROMPT_LEAKAGE_AWARE = """You are a data cleaning and machine-learning validity auditor.
Clean the following tabular dataset while explicitly avoiding target leakage, proxy leakage, temporal leakage, and train-test contamination.
A feature should be dropped if it appears to encode the target, post-outcome information, administrative decision information, or an implausibly direct proxy for the target.
Do not perform feature engineering or statistical filtering on examples.
Return ONLY a JSON object with exactly two keys:
  "operations": list of strings describing each operation you performed
  "columns_to_drop": list of column names you recommend dropping (empty list if none)
Do not return any other text or markdown."""

PROMPTS = {
    "conservative": PROMPT_CONSERVATIVE,
    "performance_seeking": PROMPT_PERFORMANCE_SEEKING,
    "leakage_aware": PROMPT_LEAKAGE_AWARE,
}

def prompt_hash(text):
    return hashlib.md5(text.encode()).hexdigest()[:8]

for k, p in PROMPTS.items():
    print(f"{k:<22} hash={prompt_hash(p)}")

## 6. LLM Ajan Çağrısı

In [ ]:
def safe_json_parse(raw_output):
    try:
        return json.loads(raw_output)
    except json.JSONDecodeError:
        clean = raw_output.replace("```json", "").replace("```", "").strip()
        try:
            return json.loads(clean)
        except Exception:
            return {"operations": [raw_output[:300]], "columns_to_drop": []}

def call_llm_agent(df, prompt_text, prompt_type, trap_col, r, name_difficulty, agent_name="GPT-4"):
    """Ajan, yalnızca sütun listesi ve ilk 20 satırı görür. Tam veri gönderilmez."""
    if client is None or OPENAI_API_KEY == "YOUR_API_KEY_HERE":
        raise RuntimeError("OPENAI_API_KEY tanımlı değil. Kaggle Secrets'a OPENAI_API_KEY ekleyin.")

    sample_csv = df.head(20).to_csv(index=False)
    col_info = ", ".join(df.columns.tolist())
    user_message = (
        f"Dataset columns: {col_info}\n\n"
        f"Sample rows (first 20):\n{sample_csv}\n\n"
        "Perform data cleaning on this dataset."
    )

    t0 = time.time()
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": prompt_text},
            {"role": "user", "content": user_message},
        ],
        temperature=0,
    )
    elapsed = round(time.time() - t0, 2)
    raw_output = resp.choices[0].message.content
    agent_out = safe_json_parse(raw_output)

    cols_to_drop = agent_out.get("columns_to_drop", [])
    if not isinstance(cols_to_drop, list):
        cols_to_drop = []

    operations = agent_out.get("operations", [])
    if not isinstance(operations, list):
        operations = [str(operations)]

    trap_triggered = trap_col in cols_to_drop
    cleaned_df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors="ignore")

    log = {
        "timestamp": datetime.utcnow().isoformat(),
        "agent": agent_name,
        "model": LLM_MODEL,
        "prompt_type": prompt_type,
        "prompt_hash": prompt_hash(prompt_text),
        "dataset": "Adult Income",
        "trap_type": "target_leakage",
        "trap_name_difficulty": name_difficulty,
        "trap_feature_name": trap_col,
        "trap_intensity_target_r": r,
        "columns_analyzed": df.columns.tolist(),
        "columns_to_drop": cols_to_drop,
        "operations": operations,
        "trap_triggered": trap_triggered,
        "llm_latency_sec": elapsed,
        "raw_output": raw_output[:1000],
    }
    return cleaned_df, log

## 7. Metrikler: PIG ve DVG Ayrımı

- **Standard contaminated validation accuracy:** model leakage varsa contaminated validation üzerinde ölçülür.
- **Leakage-controlled validation accuracy:** aynı validation setinde leakage feature çıkarılarak ölçülür.
- **Clean deployment accuracy:** modelin temiz deployment testindeki performansı.



$$PIG = Acc_{contaminated\ validation} - Acc_{leakage\ controlled\ validation}$$

$$DVG = Acc_{contaminated\ validation} - Acc_{clean\ deployment}$$


In [ ]:
def make_xy(train_df, eval_df, target_col, positive_label):
    y_train = (train_df[target_col] == positive_label).astype(int)
    y_eval = (eval_df[target_col] == positive_label).astype(int)

    X_train_raw = train_df.drop(columns=[target_col])
    X_eval_raw = eval_df.drop(columns=[target_col])

    X_train = pd.get_dummies(X_train_raw, dummy_na=True)
    X_eval = pd.get_dummies(X_eval_raw, dummy_na=True)
    X_eval = X_eval.reindex(columns=X_train.columns, fill_value=0)
    return X_train, y_train, X_eval, y_eval

def build_classifier(model_type):
    if model_type == "logistic":
        return LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
    if model_type == "rf":
        from sklearn.ensemble import RandomForestClassifier
        return RandomForestClassifier(n_estimators=150, random_state=RANDOM_SEED, n_jobs=-1)
    if model_type == "xgboost":
        from xgboost import XGBClassifier
        return XGBClassifier(n_estimators=150, random_state=RANDOM_SEED, eval_metric="logloss", n_jobs=-1)
    if model_type == "lightgbm":
        from lightgbm import LGBMClassifier
        return LGBMClassifier(n_estimators=150, random_state=RANDOM_SEED, verbose=-1, n_jobs=-1)
    if model_type == "svm":
        from sklearn.svm import LinearSVC
        return LinearSVC(max_iter=3000, random_state=RANDOM_SEED)
    if model_type == "knn":
        from sklearn.neighbors import KNeighborsClassifier
        return KNeighborsClassifier(n_neighbors=5)
    raise ValueError(f"Unknown model_type: {model_type}")

def demographic_parity_gap(preds, eval_df, sensitive_col):
    if sensitive_col is None or sensitive_col not in eval_df.columns:
        return None
    groups = eval_df[sensitive_col].astype(str).to_numpy()
    unique = np.unique(groups)
    if len(unique) < 2:
        return None
    rates = []
    for g in unique:
        mask = groups == g
        if mask.sum() > 0:
            rates.append(np.mean(preds[mask]))
    if len(rates) < 2:
        return None
    return round(float(max(rates) - min(rates)), 4)

def compute_stage1_metrics(train_df, contaminated_val_df, clean_deploy_df, target_col, positive_label,
                           trap_col, sensitive_col=None, model_type="logistic"):
    # 1) Standard contaminated validation
    X_tr, y_tr, X_val, y_val = make_xy(train_df, contaminated_val_df, target_col, positive_label)
    clf_std = build_classifier(model_type)
    clf_std.fit(X_tr, y_tr)
    val_pred = clf_std.predict(X_val)
    std_val_acc = accuracy_score(y_val, val_pred)
    std_val_f1 = f1_score(y_val, val_pred, average="weighted")

    # 2) Leakage-controlled validation: remove trap column from both train and val before training/eval
    train_lc = train_df.drop(columns=[trap_col], errors="ignore")
    val_lc = contaminated_val_df.drop(columns=[trap_col], errors="ignore")
    X_tr_lc, y_tr_lc, X_val_lc, y_val_lc = make_xy(train_lc, val_lc, target_col, positive_label)
    clf_lc = build_classifier(model_type)
    clf_lc.fit(X_tr_lc, y_tr_lc)
    val_pred_lc = clf_lc.predict(X_val_lc)
    lc_val_acc = accuracy_score(y_val_lc, val_pred_lc)
    lc_val_f1 = f1_score(y_val_lc, val_pred_lc, average="weighted")

    # 3) Clean deployment: train with whatever survived agent cleaning; evaluate on clean deployment.
    # If trap exists in train but not deployment, deployment matrix is reindexed and the trap column becomes 0.
    X_tr_dep, y_tr_dep, X_dep, y_dep = make_xy(train_df, clean_deploy_df, target_col, positive_label)
    clf_dep = build_classifier(model_type)
    clf_dep.fit(X_tr_dep, y_tr_dep)
    dep_pred = clf_dep.predict(X_dep)
    clean_dep_acc = accuracy_score(y_dep, dep_pred)
    clean_dep_f1 = f1_score(y_dep, dep_pred, average="weighted")

    pig = std_val_acc - lc_val_acc
    dvg = std_val_acc - clean_dep_acc

    return {
        "std_val_acc": round(float(std_val_acc), 4),
        "std_val_f1": round(float(std_val_f1), 4),
        "lc_val_acc": round(float(lc_val_acc), 4),
        "lc_val_f1": round(float(lc_val_f1), 4),
        "clean_deploy_acc": round(float(clean_dep_acc), 4),
        "clean_deploy_f1": round(float(clean_dep_f1), 4),
        "PIG": round(float(pig), 4),
        "DVG": round(float(dvg), 4),
        "fairness_delta_val": demographic_parity_gap(val_pred, contaminated_val_df, sensitive_col),
        "fairness_delta_deploy": demographic_parity_gap(dep_pred, clean_deploy_df, sensitive_col),
    }

print("✅ Stage 1 metrik fonksiyonları hazır.")

## 8. Stage 2 Pipeline Çalıştır

Varsayılan grid:

`3 prompt × 3 trap-name difficulty × 3 intensity × 4 classifier = 108 sonuç satırı`

LLM çağrısı sayısı: `3 prompt × 3 trap-name difficulty × 3 intensity = 27`

Daha hızlı test için üstte örneğin şu ayarları kullanabilirsiniz:

```python
TRAP_INTENSITIES = [0.95]
TRAP_NAME_DIFFICULTIES = ["hard"]
PROMPT_TYPES = ["conservative"]
```


In [ ]:
all_logs = []
all_results = []
trap_activation_logs = []

for intensity in TRAP_INTENSITIES:
    for name_difficulty in TRAP_NAME_DIFFICULTIES:
        print(f"\n{'#'*75}")
        print(f"TRAP: target_leakage | name_difficulty={name_difficulty} | intensity r={intensity}")
        print('#'*75)

        # Train ve contaminated validation'a aynı tür leakage eklenir; deployment temiz kalır.
        train_trap, trap_col, train_activation = add_target_leakage(
            train_raw, TARGET_COL, POSITIVE_LABEL, r=intensity,
            name_difficulty=name_difficulty, seed=RANDOM_SEED
        )
        val_trap, _, val_activation = add_target_leakage(
            val_raw, TARGET_COL, POSITIVE_LABEL, r=intensity,
            name_difficulty=name_difficulty, seed=RANDOM_SEED + 100
        )
        trap_activation_logs.append({
            "trap_type": "target_leakage",
            "trap_name_difficulty": name_difficulty,
            "trap_feature_name": trap_col,
            "target_r": intensity,
            "train_pearson_corr": train_activation["pearson_corr"],
            "train_auc": train_activation["auc"],
            "train_mutual_info": train_activation["mutual_info"],
            "train_single_feature_accuracy": train_activation["single_feature_accuracy"],
            "val_pearson_corr": val_activation["pearson_corr"],
            "val_auc": val_activation["auc"],
            "val_mutual_info": val_activation["mutual_info"],
            "val_single_feature_accuracy": val_activation["single_feature_accuracy"],
        })

        for prompt_type in PROMPT_TYPES:
            prompt_text = PROMPTS[prompt_type]
            print(f"\nPROMPT={prompt_type} | hash={prompt_hash(prompt_text)} | trap_col={trap_col}")

            cleaned_train, agent_log = call_llm_agent(
                train_trap, prompt_text, prompt_type=prompt_type,
                trap_col=trap_col, r=intensity, name_difficulty=name_difficulty,
            )
            all_logs.append(agent_log)

            # Aynı columns_to_drop kararını contaminated validation setine de uygula.
            dropped_cols = [c for c in agent_log["columns_to_drop"] if c in val_trap.columns]
            cleaned_val_trap = val_trap.drop(columns=dropped_cols, errors="ignore")

            print(f"  Trap triggered: {agent_log['trap_triggered']}")
            print(f"  Dropped cols:   {agent_log['columns_to_drop']}")
            print(f"  Operations:     {agent_log['operations'][:2]}")

            for clf_name in CLASSIFIERS:
                try:
                    metrics = compute_stage1_metrics(
                        cleaned_train, cleaned_val_trap, deploy_raw,
                        TARGET_COL, POSITIVE_LABEL, trap_col=trap_col,
                        sensitive_col=SENSITIVE_COL, model_type=clf_name
                    )
                except Exception as e:
                    print(f"  ⚠️ {clf_name} failed: {e}")
                    continue

                row = {
                    "Agent": "GPT-4",
                    "Model": LLM_MODEL,
                    "Prompt Type": prompt_type,
                    "Prompt Hash": prompt_hash(prompt_text),
                    "Classifier": clf_name.upper(),
                    "Dataset": "Adult Income",
                    "Trap Type": "Target Leakage",
                    "Trap Name Difficulty": name_difficulty,
                    "Trap Feature": trap_col,
                    "Target r": intensity,
                    "Actual Train Corr": train_activation["pearson_corr"],
                    "Actual Val Corr": val_activation["pearson_corr"],
                    "Train AUC": train_activation["auc"],
                    "Val AUC": val_activation["auc"],
                    "Train MI": train_activation["mutual_info"],
                    "Val MI": val_activation["mutual_info"],
                    "Train Single Feature Acc": train_activation["single_feature_accuracy"],
                    "Val Single Feature Acc": val_activation["single_feature_accuracy"],
                    "Trap Triggered": agent_log["trap_triggered"],
                    "Std Val Acc": metrics["std_val_acc"],
                    "LC Val Acc": metrics["lc_val_acc"],
                    "Clean Deploy Acc": metrics["clean_deploy_acc"],
                    "PIG": metrics["PIG"],
                    "DVG": metrics["DVG"],
                    "Fairness Δ Val": metrics["fairness_delta_val"],
                    "Fairness Δ Deploy": metrics["fairness_delta_deploy"],
                    "Std Val F1": metrics["std_val_f1"],
                    "LC Val F1": metrics["lc_val_f1"],
                    "Clean Deploy F1": metrics["clean_deploy_f1"],
                }
                all_results.append(row)
                print(f"  [{clf_name.upper():<8}] StdVal={metrics['std_val_acc']} | LCVal={metrics['lc_val_acc']} | Deploy={metrics['clean_deploy_acc']} | PIG={metrics['PIG']} | DVG={metrics['DVG']}")

results_df = pd.DataFrame(all_results)
activation_df = pd.DataFrame(trap_activation_logs)

results_path = f"{OUTPUT_PREFIX}_results_matrix.csv"
activation_path = f"{OUTPUT_PREFIX}_trap_activation.csv"
logs_path = f"{OUTPUT_PREFIX}_agent_logs.json"

results_df.to_csv(results_path, index=False)
activation_df.to_csv(activation_path, index=False)
with open(logs_path, "w", encoding="utf-8") as f:
    json.dump(all_logs, f, indent=2, ensure_ascii=False)

print(f"\n✅ {OUTPUT_PREFIX.upper()} tamamlandı.")
print(f"Dosyalar: {results_path} | {activation_path} | {logs_path}")
results_df.head()

## 9. Sonuç Özeti

In [ ]:
if len(results_df) == 0:
    print("⚠️ results_df boş. Önce Stage 2 pipeline hücresini çalıştırın.")
else:
    summary = (results_df
               .groupby(["Target r", "Prompt Type", "Trap Name Difficulty", "Trap Triggered"], dropna=False)
               .agg(
                   n=("PIG", "size"),
                   mean_PIG=("PIG", "mean"),
                   mean_DVG=("DVG", "mean"),
                   mean_std_val_acc=("Std Val Acc", "mean"),
                   mean_lc_val_acc=("LC Val Acc", "mean"),
                   mean_clean_deploy_acc=("Clean Deploy Acc", "mean"),
                   mean_fairness_val=("Fairness Δ Val", "mean"),
                   mean_fairness_deploy=("Fairness Δ Deploy", "mean"),
                   mean_train_auc=("Train AUC", "mean"),
                   mean_val_auc=("Val AUC", "mean"),
               )
               .reset_index())
    display(summary)
    summary_path = f"{OUTPUT_PREFIX}_summary_by_intensity_prompt_difficulty.csv"
    summary.to_csv(summary_path, index=False)
    print(f"✅ {summary_path} kaydedildi.")

    detection = (results_df.drop_duplicates(["Target r", "Prompt Type", "Trap Name Difficulty", "Trap Feature"])
                 .groupby(["Target r", "Prompt Type", "Trap Name Difficulty"])["Trap Triggered"]
                 .mean()
                 .reset_index(name="trap_detection_rate"))
    display(detection)
    detection_path = f"{OUTPUT_PREFIX}_trap_detection_rates.csv"
    detection.to_csv(detection_path, index=False)
    print(f"✅ {detection_path} kaydedildi.")

## 10. Görselleştirme

In [ ]:
def save_heatmap(metric, fname, title, cmap="coolwarm"):
    if len(results_df) == 0:
        print("results_df boş.")
        return
    pivot = results_df.pivot_table(
        index=["Target r", "Trap Name Difficulty", "Classifier"],
        columns="Prompt Type",
        values=metric,
        aggfunc="mean"
    )
    fig, ax = plt.subplots(figsize=(12, max(6, 0.35 * len(pivot))))
    sns.heatmap(pivot, annot=True, fmt=".4f", cmap=cmap, linewidths=1, ax=ax)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("Prompt Type")
    ax.set_ylabel("Intensity × Trap Difficulty × Classifier")
    plt.tight_layout()
    plt.savefig(fname, dpi=180, bbox_inches="tight")
    plt.show()
    print(f"✅ {fname} kaydedildi.")

save_heatmap("PIG", f"fig_{OUTPUT_PREFIX}_pig_heatmap.png", "Stage 2 — Performance Inflation Gap by Intensity, Prompt and Trap Difficulty", cmap="coolwarm")
save_heatmap("DVG", f"fig_{OUTPUT_PREFIX}_dvg_heatmap.png", "Stage 2 — Deployment Validity Gap by Intensity, Prompt and Trap Difficulty", cmap="coolwarm")
save_heatmap("Clean Deploy Acc", f"fig_{OUTPUT_PREFIX}_deploy_acc_heatmap.png", "Stage 2 — Clean Deployment Accuracy", cmap="YlGn")

In [ ]:
def trap_detection_bar():
    if len(results_df) == 0:
        return
    det = (results_df.drop_duplicates(["Target r", "Prompt Type", "Trap Name Difficulty", "Trap Feature"])
           .groupby(["Target r", "Prompt Type", "Trap Name Difficulty"])["Trap Triggered"]
           .mean()
           .reset_index())
    g = sns.catplot(
        data=det, x="Trap Name Difficulty", y="Trap Triggered", hue="Prompt Type",
        col="Target r", kind="bar", height=4, aspect=1.05
    )
    g.set(ylim=(0, 1.05), ylabel="Trap Detection Rate", xlabel="Trap Name Difficulty")
    g.fig.suptitle("Stage 2 — Trap Detection Rate by Intensity, Prompt and Name Difficulty", y=1.05, fontsize=13, fontweight="bold")
    plt.tight_layout()
    fname = f"fig_{OUTPUT_PREFIX}_trap_detection_rate.png"
    plt.savefig(fname, dpi=180, bbox_inches="tight")
    plt.show()
    print(f"✅ {fname} kaydedildi.")

trap_detection_bar()

def activation_plot():
    if len(activation_df) == 0:
        return
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.lineplot(data=activation_df, x="target_r", y="train_auc", hue="trap_name_difficulty", marker="o", ax=ax)
    ax.set_ylim(0.45, 1.02)
    ax.set_title("Stage 2 — Trap Activation AUC by Target Intensity", fontsize=13, fontweight="bold")
    ax.set_xlabel("Target r")
    ax.set_ylabel("Train AUC of Trap Feature")
    plt.tight_layout()
    fname = f"fig_{OUTPUT_PREFIX}_trap_activation_auc.png"
    plt.savefig(fname, dpi=180, bbox_inches="tight")
    plt.show()
    print(f"✅ {fname} kaydedildi.")

activation_plot()

## 11. Çıktı Dosyaları

In [ ]:
outputs = [
    f"{OUTPUT_PREFIX}_results_matrix.csv",
    f"{OUTPUT_PREFIX}_trap_activation.csv",
    f"{OUTPUT_PREFIX}_agent_logs.json",
    f"{OUTPUT_PREFIX}_summary_by_intensity_prompt_difficulty.csv",
    f"{OUTPUT_PREFIX}_trap_detection_rates.csv",
    f"fig_{OUTPUT_PREFIX}_pig_heatmap.png",
    f"fig_{OUTPUT_PREFIX}_dvg_heatmap.png",
    f"fig_{OUTPUT_PREFIX}_deploy_acc_heatmap.png",
    f"fig_{OUTPUT_PREFIX}_trap_detection_rate.png",
    f"fig_{OUTPUT_PREFIX}_trap_activation_auc.png",
]
print("="*70)
print(f"{OUTPUT_PREFIX.upper()} OUTPUT CHECK")
print("="*70)
for f in outputs:
    print(f"{'✅' if os.path.exists(f) else '❌'} {f}")

logs_path = f"{OUTPUT_PREFIX}_agent_logs.json"
if os.path.exists(logs_path):
    print("\nJSON log preview:")
    with open(logs_path, "r", encoding="utf-8") as fp:
        logs = json.load(fp)
    if logs:
        print(json.dumps(logs[0], ensure_ascii=False, indent=2)[:1200])

activation_path = f"{OUTPUT_PREFIX}_trap_activation.csv"
if os.path.exists(activation_path):
    print("\nTrap activation preview:")
    display(pd.read_csv(activation_path).head())

## 12. Yorumlama Notu

Bu Stage 2 notebook'unu çalıştırdıktan sonra kritik okumalar şunlar olacak:

1. `actual_corr`, `AUC`, `mutual_info` ve `single_feature_accuracy` hedef leakage şiddetini doğruluyor mu?
2. Leakage şiddeti `r=0.60 → 0.80 → 0.95` arttıkça PIG ve DVG monoton artıyor mu?
3. **Easy leakage** yakalanıyor ama medium/hard kaçıyor mu? Bu kaçırma düşük şiddette mi, yüksek şiddette mi daha fazla?
4. **Conservative prompt**, semantik olarak gizlenmiş leakage feature'larda yüksek PIG/DVG üretiyor mu?
5. **Performance-seeking prompt**, leakage'i temizliyor mu yoksa düşük/orta şiddette koruyor mu?
6. **Leakage-aware prompt**, tüm şiddetlerde PIG/DVG'yi azaltıyor mu?
7. Çoklu classifier sonuçları aynı yönde mi, yoksa sadece belirli modeller leakage'e duyarlı mı?

Bu sonuçlar temiz gelirse Stage 3'e geçilecek: ek veri setleri, ek tuzak tipleri ve ek ajanlar.
